# Review Dataset Segregator

Extract the product flavor from the `Model` column and write it into a new `Flavor` column in the Excel workbook.


In [ ]:
pip install openpyxl

In [2]:
from pathlib import Path
import re

import pandas as pd


In [3]:
FLAVOR_LABELS = ("Flavor Name", "Parfum", "Nome sapore")
STOP_LABELS = ("Size", "Taille", "Taglia")

flavor_pattern = re.compile(
    rf"(?:{'|'.join(re.escape(label) for label in FLAVOR_LABELS)})\s*:\s*(.+?)"
    rf"(?=(?:{'|'.join(re.escape(label) for label in STOP_LABELS)})\s*:|$)",
    re.IGNORECASE,
)


def extract_flavor(model_value):
    if pd.isna(model_value):
        return pd.NA

    model_text = str(model_value).strip()
    if not model_text:
        return pd.NA

    match = flavor_pattern.search(model_text)
    if match:
        flavor = re.sub(r"\s+", " ", match.group(1)).strip()
        return flavor or pd.NA

    if ":" not in model_text:
        return model_text

    return pd.NA


In [7]:
input_path = Path("D:\\hassan_manager\\bone_broth\\Bone_Broth_reviews.xlsx")
# Change this to `input_path` if you want to overwrite the source workbook.
output_path = input_path.with_name(f"{input_path.stem}_with_flavor{input_path.suffix}")

sheet_frames = pd.read_excel(input_path, sheet_name=None)

for sheet_name, df in sheet_frames.items():
    if "Model" not in df.columns:
        continue

    flavor_values = df["Model"].apply(extract_flavor)

    if "Flavor" in df.columns:
        df["Flavor"] = flavor_values
    else:
        insert_at = df.columns.get_loc("Model") + 1
        df.insert(insert_at, "Flavor", flavor_values)

with pd.ExcelWriter(output_path) as writer:
    for sheet_name, df in sheet_frames.items():
        df.to_excel(writer, sheet_name=sheet_name, index=False)

print(f"Saved updated workbook to: {output_path.resolve()}")

first_sheet_name, first_sheet_df = next(iter(sheet_frames.items()))
preview_columns = [column for column in ("Brand", "Model", "Flavor") if column in first_sheet_df.columns]
first_sheet_df[preview_columns].head(10)


Saved updated workbook to: D:\hassan_manager\bone_broth\Bone_Broth_reviews_with_flavor.xlsx


,Brand,Model,Flavor
0,Paleovalley,Flavor Name:ChocolateSize:1.18 Pound (Pack of 1),Chocolate
1,Paleovalley,Flavor Name:UnflavoredSize:1.05 Pound (Pack of 1),Unflavored
2,Paleovalley,Flavor Name:UnflavoredSize:1.05 Pound (Pack of 1),Unflavored
3,Paleovalley,Flavor Name:ChocolateSize:1.18 Pound (Pack of 1),Chocolate
4,Paleovalley,Flavor Name:UnflavoredSize:1.05 Pound (Pack of 1),Unflavored
5,Paleovalley,Flavor Name:ChocolateSize:1.18 Pound (Pack of 1),Chocolate
6,Paleovalley,Flavor Name:VanillaSize:1.17 Pound (Pack of 1),Vanilla
7,Paleovalley,Flavor Name:ChocolateSize:1.18 Pound (Pack of 1),Chocolate
8,Paleovalley,Flavor Name:UnflavoredSize:1.05 Pound (Pack of 1),Unflavored
9,Paleovalley,Flavor Name:UnflavoredSize:1.05 Pound (Pack of 1),Unflavored


In [ ]:
unmatched_models = {}

for sheet_name, df in sheet_frames.items():
    if "Model" not in df.columns or "Flavor" not in df.columns:
        continue

    missing_flavor = df["Model"].notna() & df["Flavor"].isna()
    if missing_flavor.any():
        unmatched_models[sheet_name] = sorted(df.loc[missing_flavor, "Model"].astype(str).unique())

unmatched_models
